In [1]:
# Detailed test fire label audit
# Checks EVERY 2-day window in center 256x256 crop -- exactly what inference sees

import os, glob, rasterio
import numpy as np

DATA_ROOT = "/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire"
TS = 2
PATCH = 256

AF_TEST_FIRES = [
    "elephant_hill_fire", "eagle_bluff_fire", "double_creek_fire",
    "sparks_lake_fire", "lytton_fire", "chuckegg_creek_fire",
    "swedish_fire", "sydney_fire", "thomas_fire", "tubbs_fire",
    "carr_fire", "camp_fire", "creek_fire", "blue_ridge_fire",
    "dixie_fire", "mosquito_fire", "calfcanyon_fire",
]

print(f"Checking all test fires: every {TS}-day window, center {PATCH}x{PATCH} crop")
print(f"{'='*90}")
print(f"{'Fire':<25s} {'Days':>4} {'Windows':>7} {'W/Label':>7} {'W/Fire':>7} "
      f"{'TotFirePx':>10} {'Status':<20s}")
print(f"{'-'*90}")

usable_fires = []
partial_fires = []
no_label_fires = []

for fid in sorted(AF_TEST_FIRES):
    fdir = os.path.join(DATA_ROOT, fid)
    day_tifs = sorted(glob.glob(os.path.join(fdir, "VIIRS_Day", "*.tif")))
    n_days = len(day_tifs)

    if n_days < TS:
        print(f"{fid:<25s} {n_days:>4}  TOO FEW DAYS")
        no_label_fires.append(fid)
        continue

    # Get image size from first file
    with rasterio.open(day_tifs[0]) as src:
        H, W = src.height, src.width
    r0 = (H - PATCH) // 2
    c0 = (W - PATCH) // 2

    n_windows = 0
    windows_with_label = 0
    windows_with_fire = 0
    total_fire_px = 0
    per_day_status = []

    # Check each day individually first
    for tif in day_tifs:
        with rasterio.open(tif) as src:
            if src.count >= 7:
                b7 = src.read(7).astype(np.float32)
                all_nan = np.isnan(b7).sum() == b7.size
                if not all_nan:
                    crop = b7[r0:r0+PATCH, c0:c0+PATCH]
                    fire_px = int((crop >= 7).sum())
                    per_day_status.append(("OK", fire_px))
                else:
                    per_day_status.append(("NaN", 0))
            else:
                per_day_status.append(("NoBand7", 0))

    # Check each 2-day window (same sliding as inference)
    for t0 in range(n_days - TS + 1):
        n_windows += 1
        last_day = day_tifs[t0 + TS - 1]

        with rasterio.open(last_day) as src:
            if src.count >= 7:
                b7 = src.read(7).astype(np.float32)
                all_nan = np.isnan(b7).sum() == b7.size
                if not all_nan:
                    windows_with_label += 1
                    crop = b7[r0:r0+PATCH, c0:c0+PATCH]
                    fire_px = int((crop >= 7).sum())
                    total_fire_px += fire_px
                    if fire_px > 0:
                        windows_with_fire += 1

    # Classify
    if windows_with_label == 0:
        status = "NO LABELS"
        no_label_fires.append(fid)
    elif windows_with_fire == 0:
        status = "LABELS BUT NO FIRE"
        partial_fires.append(fid)
    elif windows_with_label < n_windows * 0.5:
        status = f"PARTIAL ({windows_with_label}/{n_windows})"
        usable_fires.append(fid)
    else:
        status = "GOOD"
        usable_fires.append(fid)

    print(f"{fid:<25s} {n_days:>4} {n_windows:>7} {windows_with_label:>7} "
          f"{windows_with_fire:>7} {total_fire_px:>10}  {status}")

    # Show per-day detail for problematic fires
    if windows_with_fire == 0 or windows_with_label < n_windows * 0.5:
        for d, (st, fp) in enumerate(per_day_status):
            print(f"  Day {d+1}: {st} fire_px={fp} ({os.path.basename(day_tifs[d])})")

print(f"\n{'='*90}")
print(f"SUMMARY:")
print(f"  GOOD fires (usable for eval): {len(usable_fires)}")
print(f"    {usable_fires}")
print(f"  PARTIAL (some labels, check carefully): {len(partial_fires)}")
print(f"    {partial_fires}")
print(f"  NO LABELS (exclude from eval): {len(no_label_fires)}")
print(f"    {no_label_fires}")
print(f"\nFor fair paper comparison, evaluate on the {len(usable_fires)} good fires only.")

Checking all test fires: every 2-day window, center 256x256 crop
Fire                      Days Windows W/Label  W/Fire  TotFirePx Status              
------------------------------------------------------------------------------------------
blue_ridge_fire             10       9       9       9        731  GOOD
calfcanyon_fire             10       9       0       0          0  NO LABELS
  Day 1: NaN fire_px=0 (2022-04-05_VIIRS_Day.tif)
  Day 2: NaN fire_px=0 (2022-04-06_VIIRS_Day.tif)
  Day 3: NaN fire_px=0 (2022-04-07_VIIRS_Day.tif)
  Day 4: NaN fire_px=0 (2022-04-08_VIIRS_Day.tif)
  Day 5: NaN fire_px=0 (2022-04-09_VIIRS_Day.tif)
  Day 6: NaN fire_px=0 (2022-04-10_VIIRS_Day.tif)
  Day 7: NaN fire_px=0 (2022-04-11_VIIRS_Day.tif)
  Day 8: NaN fire_px=0 (2022-04-12_VIIRS_Day.tif)
  Day 9: NaN fire_px=0 (2022-04-13_VIIRS_Day.tif)
  Day 10: NaN fire_px=0 (2022-04-14_VIIRS_Day.tif)
camp_fire                   10       9       9       9       6470  GOOD
carr_fire                   10     